# MiniViT on CIFAR-10

This notebook builds a small Vision Transformer (ViT) from PyTorch components and trains it to classify CIFAR-10 images. A 32×32 image is split into 4×4 patches, producing 64 patch tokens. A learnable class token summarizes the image after the Transformer encoder.

In [1]:
from pathlib import Path
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"PyTorch: {torch.__version__} | device: {device}")

ImportError: dlopen(/Users/minghanli/miniconda3/lib/python3.13/site-packages/torch/_C.cpython-313-darwin.so, 0x0002): Symbol not found: __ZN2at17toDLPackVersionedERKNS_6TensorE
  Referenced from: <1DE41B0A-6593-33CF-B238-A0A59BBE0166> /Users/minghanli/miniconda3/lib/python3.13/site-packages/torch/lib/libtorch_python.dylib
  Expected in:     <A487EBF3-4C9D-373E-8235-C7A4DCA6725E> /Users/minghanli/miniconda3/lib/libtorch_cpu.dylib

## Load CIFAR-10

Training images use random crops and horizontal flips. Both splits are normalized with CIFAR-10 channel statistics. Set `QUICK_RUN = True` to test the pipeline for one epoch on a subset before running the full experiment.

In [ ]:
QUICK_RUN = False
BATCH_SIZE = 128
NUM_WORKERS = 2
EPOCHS = 1 if QUICK_RUN else 20

# Resolve paths whether the notebook is launched from the repository root or its own folder.
DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
CHECKPOINT_DIR = Path("output")
CHECKPOINT_DIR.mkdir(exist_ok=True)

mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_dataset = datasets.CIFAR10(DATA_DIR, train=True, download=True, transform=train_transform)
test_dataset = datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=test_transform)
if QUICK_RUN:
    train_dataset = torch.utils.data.Subset(train_dataset, range(4096))
    test_dataset = torch.utils.data.Subset(test_dataset, range(1024))

loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=device.type == "cuda")
train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)
class_names = datasets.CIFAR10(DATA_DIR, train=False, download=False).classes
print(f"Training images: {len(train_dataset):,} | test images: {len(test_dataset):,}")
print(class_names)

In [ ]:
# Display unnormalized examples.
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for image, label, ax in zip(images[:12], labels[:12], axes.flat):
    image = image.permute(1, 2, 0) * torch.tensor(std) + torch.tensor(mean)
    ax.imshow(image.clamp(0, 1))
    ax.set_title(class_names[label])
    ax.axis("off")
plt.tight_layout()

## MiniViT model

A strided convolution performs patch extraction and linear projection in one operation. Learnable position embeddings preserve spatial order. The encoder then repeatedly applies multi-head self-attention and feed-forward layers.

In [ ]:
class MiniViT(nn.Module):
    def __init__(
        self, image_size=32, patch_size=4, num_classes=10, embed_dim=192,
        depth=6, num_heads=6, mlp_ratio=4, dropout=0.1
    ):
        super().__init__()
        assert image_size % patch_size == 0
        num_patches = (image_size // patch_size) ** 2
        self.patch_embed = nn.Conv2d(3, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim * mlp_ratio,
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth, norm=nn.LayerNorm(embed_dim))
        self.head = nn.Linear(embed_dim, num_classes)
        self.apply(self._init_weights)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Conv2d)):
            nn.init.trunc_normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, x):
        x = self.patch_embed(x).flatten(2).transpose(1, 2)  # (batch, patches, embedding)
        cls = self.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls, x), dim=1)
        x = self.pos_dropout(x + self.pos_embed)
        x = self.encoder(x)
        return self.head(x[:, 0])

model = MiniViT().to(device)
parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
with torch.no_grad():
    output_shape = model(torch.randn(2, 3, 32, 32, device=device)).shape
print(model)
print(f"Trainable parameters: {parameter_count / 1e6:.2f}M | output shape: {tuple(output_shape)}")

## Train and evaluate

AdamW is commonly used for Transformers. A cosine schedule gradually reduces the learning rate, while gradient clipping protects against unstable updates. The checkpoint with the best test accuracy is saved.

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def run_epoch(model, loader, training):
    model.train(training)
    total_loss = total_correct = total_examples = 0
    context = torch.enable_grad() if training else torch.inference_mode()
    with context:
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(inputs)
            loss = criterion(logits, targets)
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            total_loss += loss.item() * targets.size(0)
            total_correct += (logits.argmax(dim=1) == targets).sum().item()
            total_examples += targets.size(0)
    return total_loss / total_examples, total_correct / total_examples

history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
best_accuracy = 0.0
checkpoint_path = CHECKPOINT_DIR / "minivit_cifar10_best.pt"

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    train_loss, train_acc = run_epoch(model, train_loader, training=True)
    test_loss, test_acc = run_epoch(model, test_loader, training=False)
    scheduler.step()
    for key, value in zip(history, (train_loss, train_acc, test_loss, test_acc)):
        history[key].append(value)
    if test_acc > best_accuracy:
        best_accuracy = test_acc
        torch.save({"model_state_dict": model.state_dict(), "test_accuracy": test_acc}, checkpoint_path)
    print(
        f"Epoch {epoch:02d}/{EPOCHS} | {time.time() - start:5.1f}s | "
        f"train loss {train_loss:.4f}, acc {train_acc:.2%} | "
        f"test loss {test_loss:.4f}, acc {test_acc:.2%}"
    )

print(f"Best test accuracy: {best_accuracy:.2%}")
print(f"Saved checkpoint: {checkpoint_path.resolve()}")

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs, history["train_loss"], label="train")
axes[0].plot(epochs, history["test_loss"], label="test")
axes[0].set(xlabel="epoch", ylabel="cross-entropy loss", title="Loss")
axes[1].plot(epochs, np.array(history["train_acc"]) * 100, label="train")
axes[1].plot(epochs, np.array(history["test_acc"]) * 100, label="test")
axes[1].set(xlabel="epoch", ylabel="accuracy (%)", title="Accuracy")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()